### SQlite에 대화 내용 저장하기

In [1]:
from langchain_community.chat_message_histories import SQLChatMessageHistory


import os

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging


load_dotenv()
logging.langsmith("test0914")


print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914


In [2]:
# SQLChatMessageHistory 객체를 생성하고 세션 ID와 데이터베이스 연결 파일을 설정
chat_message_history = SQLChatMessageHistory(
    session_id="sql_history", connection="sqlite:///sqlite.db"
)

In [3]:
# 사용자 메시지를 추가
chat_message_history.add_user_message(
    "안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!"
)

# AI 메시지를 추가
chat_message_history.add_ai_message("안녕 테디, 만나서 반가워. 나도 잘 부탁해!")

In [4]:
chat_message_history.messages

[HumanMessage(content='안녕? 만나서 반가워. 내 이름은 테디야. 나는 랭체인 개발자야. 앞으로 잘 부탁해!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕 테디, 만나서 반가워. 나도 잘 부탁해!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser


In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant."), # 시스템 메시지
        MessagesPlaceholder(variable_name="chat_history"),  # 대화 기록을 위한 Placeholder
        ("human", "{question}")  # 질문
    ]
)

In [7]:
chain = prompt | ChatOpenAI(model="gpt-4o") | StrOutputParser()

In [8]:
def get_chat_history(user_id, conversation_id):
    return SQLChatMessageHistory(
        table_name= user_id,
        session_id= conversation_id,
        connection= "sqlite:///sqlite.db"
    )

In [ ]:
from langchain_core.runnables.utils import ConfigurableFieldSpec

config_fields = [
    ConfigurableFieldSpec(
        id="user_id",  # 실제로 쓰이는 키 이름
        annotation=str,  # 타입 - 문자열
        name="User ID",  # 사람이 읽는 표시용 이름
        description="Unique identifier for a user",  # 설명 ( 문서화 용도 )
        default= "", # 값을 안 주면 쓰이는 기본값
        is_shared=True  # 여러 단계에서 공유되는 값인지
    ),
    ConfigurableFieldSpec(
        id="conversation_id",
        annotation=str,
        name="Conversation ID",
        description="Unique identifier for a conversation",
        default="",
        is_shared=True
    )
]

In [ ]:
chain_with_history = RunnableWithMessageHistory(
    chain,    # 감쌀 대상
    get_chat_history, # 기록 저장소를 돌려주는 함수  , 대화 기록을 가져오는 함수
    input_messages_key="question", # 입력 딕셔너리에서 사용자 발화가 든 키 , 입력 메시지의 키를 설정
    history_messages_key="chat_history", # 프롬프트에서 기록이 들어갈 자리 , 대화 기록 메시지의 키를 설정
    history_factory_config= config_fields # 앞서 만든 키 명세 , 대화 기록 조회시 참고할 매개변수를 설정
)

d:\kingSJ\hanwha_0902\ex_0918\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:
config = {"configurable": {"user_id": "user1", "conversation_id": "conversation1"}}

In [12]:
chain_with_history.invoke({"question":"안녕 반가워, 내 이름은 테디야"}, config)

'안녕하세요 테디! 만나서 반가워요. 어떻게 도와드릴까요?'

In [13]:
chain_with_history.invoke({"question": "내 이름이 뭐라고?"}, config)

'당신의 이름은 테디라고 하셨죠. 맞나요?'

In [15]:
# 이번에는 같은 user_id(user1)를 가지지만 conversation_id 가 다른값(conversation2)을 가지도록 하면?

config2 = {"configurable" : {"user_id":"user1", "conversation_id":"conversation2"}}

chain_with_history.invoke({"question":"내 이름이 뭐라고?"},config2)

'죄송하지만, 제가 당신의 이름을 알 수 있는 방법이 없습니다. 하지만 필요한 정보나 도움이 필요하시면 언제든지 말씀해 주세요!'